In [ ]:
pip install adjusttext

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from adjustText import adjust_text

In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/1.global-data-cba/'
ground_data_name = 'ground_data_170226_validation.gpkg'
mrt_name = 'MRT_1m.tif' 

# derived paths
ground_data_path = os.path.join(base_dir, 'MRT-prediction/ground_data', ground_data_name)
mrt_path = os.path.join(base_dir, 'MRT-prediction', mrt_name)

#### reading and calibrating ground data 

In [ ]:
gdf = gpd.read_file(ground_data_path)

In [ ]:
# calibrate TG1
# from notebook 1.4.4.1. 
gdf["tg_corr"] = np.where(
    gdf["Instrument"] == "T1 - Sper Scientific 800036 ",
    0.639 * gdf["TG"] + 9.681,   # apply correction for T1
    gdf["TG"]                    # keep original value for T2
)

#### computing observed MRT and extracting predicted MRT values for ground data

In [ ]:
# Parameters
eps = 0.95
D1 = 0.04   # diameter of globe 1 [m] 
D2 = 0.05  # diameter of globe 2 [m] 

# Select diameter according to instrument
D = np.where(gdf["Instrument"] == "T1 - Sper Scientific 800036 ", D1,D2)

# MRT calculation using row-specific diameter
gdf["MRT"] = (
    (
        (gdf["tg_corr"] + 273.15)**4
        + (1.1e8 * gdf["wind_mean"]**0.6 / (eps * D2**0.4)) * (gdf["tg_corr"] - gdf["TA"]) # use D2 because TG values are calibrated to Thermometer 2
    )**0.25
    - 273.15
)

In [ ]:
# read raster crs and sample values
with rasterio.open(mrt_path) as src:
    mrt_crs = src.crs
    points_proj = gdf.to_crs(mrt_crs)
    samples = list(src.sample(zip(points_proj.geometry.x, points_proj.geometry.y)))
    points_proj['MRT_predicted'] = [s[0] for s in samples]

points_proj['MRT_predicted'] = points_proj['MRT_predicted'].astype(float) 
points_proj["ID"] = range(len(points_proj))

# export gpkg
out_gpkg = os.path.join(base_dir, 'MRT-prediction/MRT_points_2026v2.gpkg')
points_proj.to_file(out_gpkg, driver="GPKG")


In [ ]:
obs = points_proj['MRT']
pred = points_proj['MRT_predicted']
points_proj['residuals'] = obs - pred

In [ ]:
points_proj 

#### analysis

In [ ]:
# Metrics

residuals = obs - pred

mae = np.mean(np.abs(residuals)).round(2)
rmse = np.sqrt(np.mean(residuals**2)).round(2)
bias = np.mean(residuals).round(2)

print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)

In [ ]:
# plot

# Scatter
fig = px.scatter(
    points_proj,
    x=points_proj['MRT'],
    y=points_proj['MRT_predicted'],
    color=abs(points_proj['residuals']),
    hover_data=["tg_corr", "TA","wind_mean", "Instrument", "condition", "residuals", "context"],
)

# 1:1 line
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    line=dict(color="black", width=2), 
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="observed vs predicted MRT", 
    coloraxis_colorbar=dict(title="Residuals")
)


fig.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Helper values
x = points_proj["MRT"]
y = points_proj["MRT_predicted"]

# 1) condition
for cond in points_proj["condition"].unique():
    mask = points_proj["condition"] == cond
    axes[0].scatter(x[mask], y[mask], label=cond)
axes[0].set_title("Colored by condition")
axes[0].set_xlabel("Observed MRT")
axes[0].set_ylabel("Predicted MRT")
axes[0].legend()


# 2) Instrument
for inst in points_proj["Instrument"].unique():
    mask = points_proj["Instrument"] == inst
    axes[1].scatter(x[mask], y[mask], label=inst)
axes[1].set_title("Colored by Instrument")
axes[1].set_xlabel("Observed MRT")
axes[1].set_ylabel("Predicted MRT")
axes[1].legend()


# 3) context
for ctx in points_proj["context"].unique():
    mask = points_proj["context"] == ctx
    axes[2].scatter(x[mask], y[mask], label=ctx)
axes[2].set_title("Colored by context")
axes[2].set_xlabel("Observed MRT")
axes[2].set_ylabel("Predicted MRT")
axes[2].legend()


# 1:1 reference line
for ax in axes.flat:
    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])
    ]
    ax.plot(lims, lims, color="gray")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = points_proj["MRT"]
y = points_proj["MRT_predicted"]

# 1) Condition
traduc_cond = {"sun": "Sol", "shadow": "Sombra", "SUN": "Sol", "SHADOW": "Sombra"}
colores_cond = {"Sol": "orange", "Sombra": "steelblue"}
for cond in points_proj["condition"].unique():
    mask = points_proj["condition"] == cond
    label = traduc_cond.get(cond, cond)
    axes[0].scatter(x[mask], y[mask], label=label, color=colores_cond.get(label))
axes[0].set_title("Coloreado por condición")
axes[0].set_xlabel("MRT observada (°C)")
axes[0].set_ylabel("MRT predicha (°C)")
axes[0].legend(title="Condición")

# 2) Context
traduc_ctx = {"open parking": "Estacionamiento", "park": "Parque", "residential": "Residencial"}
colores_ctx = {"Estacionamiento": "steelblue", "Parque": "green", "Residencial": "orange"}
for ctx in points_proj["context"].unique():
    mask = points_proj["context"] == ctx
    label = traduc_ctx.get(ctx, ctx)
    axes[1].scatter(x[mask], y[mask], label=label, color=colores_ctx.get(label))
axes[1].set_title("Coloreado por contexto")
axes[1].set_xlabel("MRT observada (°C)")
axes[1].set_ylabel("MRT predicha (°C)")
axes[1].legend(title="Contexto")

# 1:1 line
for ax in axes.flat:
    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])
    ]
    ax.plot(lims, lims, color="gray", linestyle="--")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter
fig = px.scatter(
    points_proj,
    x=pred,
    y=residuals,
    color="context",
    hover_data=["TG", "TA","wind_mean", "condition", "Instrument", "tg_corr", "MRT", "ID"],
)

# 1:1 line
fig.add_scatter(
    x=[pred.min(), pred.max()],
    y=[0, 0],
    mode="lines",
    name="Zero residual line"
)

fig.update_layout(
    xaxis_title="Predicted MRT °C",
    yaxis_title="Residuals (Obs - Pred)",
    title="points_proj residuals"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

traduc_ctx = {"open parking": "Estacionamiento", "park": "Parque", "residential": "Residencial"}
colors = {"open parking": "steelblue", "park": "green", "residential": "darkorange"}

for ctx in points_proj["context"].unique():
    mask = points_proj["context"] == ctx
    ax.scatter(
        pred[mask], residuals[mask],
        label=traduc_ctx.get(ctx, ctx),
        color=colors.get(ctx, "gray"),
        zorder=5
    )

# Labels with adjust_text
texts = []
for idx, row in points_proj.iterrows():
    texts.append(ax.text(pred[idx], residuals[idx], str(row["ID"]), fontsize=7))

adjust_text(texts, arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))

ax.axhline(0, color="gray", linestyle="--")
ax.set_xlabel("MRT predicha (°C)")
ax.set_ylabel("Residual (obs - pred) (°C)")
ax.set_title("Residuales por punto de medición")
ax.legend(title="Contexto")
plt.tight_layout()
plt.show()

In [ ]:
metrics_residual = (
    points_proj
    .groupby("context")["residuals"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        max="max",
        count="count",
        mae=lambda x: np.mean(np.abs(x)),
        rmse=lambda x: np.sqrt(np.mean(x**2))
    )
)

metrics_residual

In [ ]:
df_plot = points_proj.drop(columns="geometry")

fig = px.box(
    df_plot,
    x="context",
    y="residuals",
    points="all",
    hover_data=["ID"],
)

fig.add_hline(y=0, line_dash="solid", line_color="black")

fig.update_layout(
    yaxis_title="Residual (obs-pred)",
    xaxis_title="",
    xaxis_tickangle=45
)

fig.update_traces(
    jitter=0,        # 0 = centered, 1 = spread out
    pointpos=0,      # 0 = inside the box, -1 or 1 = to the side
)
fig.show()

BEFORE STARTING, LOCATION OF SSOME POINTS WAS CORRECTED. SOME OF THEM WERE UNDER TREES NOT DETECTED:ALSO MOVED UNDER A DETECTED TREE.

10 AND 4 ARE OVERESTIMATED IN MODEL BECAUSE THERE IS DETECTED A BUILDING THAT DOESNT EXIST 


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

traduc_ctx = {"open parking": "Estacionamiento", "park": "Parque", "residential": "Residencial"}

data = [points_proj[points_proj["context"] == ctx]["residuals"].dropna().values
        for ctx in traduc_ctx.keys()]

ax.boxplot(data, labels=traduc_ctx.values())
ax.axhline(0, color="black")
ax.set_ylabel("Residual (obs - pred) (°C)")
ax.set_title("Residuales por contexto")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

comparing TA with MRT

In [ ]:
df = points_proj.copy().reset_index(drop=True)


colors = {"SHADOW": "#2196F3", "SUN": "#FF5722"}

fig, ax = plt.subplots(figsize=(8, 7))

for cond, group in df.groupby("condition"):
    ax.scatter(group["TG"], group["TA"],
               color=colors.get(cond, "gray"), s=70, zorder=3,
               edgecolors="gray", linewidths=0.5, label=cond, alpha=0.85)

ax.set_xlim(df["TG"].min() - 0.5, df["TG"].max() + 0.5)
ax.set_ylim(df["TA"].min() - 0.5, df["TA"].max() + 0.5)
ax.set_aspect("equal")

ax.set_xlabel("TG (°C)")
ax.set_ylabel("TA (°C)")
ax.set_title("TA vs TG", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:


colors = {"SHADOW": "#2196F3", "SUN": "#FF5722"}

fig, ax = plt.subplots(figsize=(8, 7))

for cond, group in df.groupby("condition"):
    ax.scatter(group["MRT"], group["TA"],
               color=colors.get(cond, "gray"), s=70, zorder=3,
               edgecolors="gray", linewidths=0.5, label=cond, alpha=0.85)

ax.set_xlim(df["MRT"].min() - 0.5, df["MRT"].max() + 0.5)
ax.set_ylim(df["TA"].min() - 0.5, df["TA"].max() + 0.5)
ax.set_aspect("equal")

ax.set_xlabel("MRT (°C)")
ax.set_ylabel("TA (°C)")
ax.set_title("TA vs MRT", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for cond, group in df.groupby("context"):
    ax.scatter(group["MRT"], group["TA"],
               edgecolors="gray", linewidths=0.5, label=cond, alpha=0.85)

ax.set_xlim(df["MRT"].min() - 0.5, df["MRT"].max() + 0.5)
ax.set_ylim(df["TA"].min() - 0.5, df["TA"].max() + 0.5)
ax.set_aspect("equal")

ax.set_xlabel("MRT (°C)")
ax.set_ylabel("TA (°C)")
ax.set_title("TA vs MRT", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()